# Exploratory Data Analysis

EDA on the credit card transactions dataset. Target column: `Class` (0 = legitimate, 1 = fraud).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)

## 1. Overview

In [ ]:
DATA_PATH = Path('..') / 'data' / 'cleaned.csv'
df = pd.read_csv(DATA_PATH)

TARGET = 'Class'
TASK = 'classification' if df[TARGET].nunique() <= 20 else 'regression'

print(f'Shape: {df.shape}')
print(f'Task type: {TASK}')
print(f'\nColumns ({len(df.columns)}):')
print(list(df.columns))
print('\nDtypes:')
print(df.dtypes)
df.head()

## 2. Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

if TASK == 'classification':
    counts = df[TARGET].value_counts().sort_index()
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, palette='viridis')
    for i, v in enumerate(counts.values):
        pct = v / counts.sum() * 100
        ax.text(i, v, f'{v:,}\n({pct:.3f}%)', ha='center', va='bottom', fontsize=11)
    ax.set_title(f'Class Distribution of "{TARGET}"', fontsize=14)
    ax.set_xlabel(TARGET)
    ax.set_ylabel('Count')
    ax.set_yscale('log')
    print(counts)
    print(f'\nImbalance ratio (majority:minority) = {counts.max() / counts.min():.1f} : 1')
else:
    sns.histplot(df[TARGET], bins=50, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of Target "{TARGET}"', fontsize=14)
    ax.set_xlabel(TARGET)
    ax.set_ylabel('Frequency')
    print(df[TARGET].describe())

plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
missing = df.isnull().sum().to_frame('missing_count')
missing['missing_pct'] = missing['missing_count'] / len(df) * 100
print('Missing values per column:')
print(missing)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap (yellow = null)', fontsize=14)
ax.set_xlabel('Columns')
ax.set_ylabel('Rows')
plt.tight_layout()
plt.show()

## 4. Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [c for c in numeric_cols if c != TARGET]
features_to_plot = feature_cols[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flat, features_to_plot):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}', fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

for ax in axes.flat[len(features_to_plot):]:
    ax.set_visible(False)

fig.suptitle('Numeric Feature Distributions (first 9)', fontsize=15, y=1.00)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 7},
    cbar_kws={'shrink': 0.7},
    ax=ax,
)
ax.set_title('Correlation Matrix of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Features vs Target

In [ ]:
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top3 = target_corr.head(3).index.tolist()
print('Top 3 features by |correlation| with target:')
print(target_corr.head(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col in zip(axes, top3):
    if TASK == 'classification':
        sns.boxplot(x=df[TARGET], y=df[col], ax=ax, palette='Set2')
        ax.set_title(f'{col} by {TARGET}', fontsize=12)
        ax.set_xlabel(TARGET)
        ax.set_ylabel(col)
    else:
        ax.scatter(df[col], df[TARGET], alpha=0.4, s=10, color='steelblue')
        ax.set_title(f'{col} vs {TARGET}', fontsize=12)
        ax.set_xlabel(col)
        ax.set_ylabel(TARGET)

fig.suptitle(f'Top 3 Features vs Target ({TARGET})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Severe class imbalance**: fraud cases are a tiny minority of the dataset, so accuracy alone is misleading — evaluation should rely on precision/recall, ROC-AUC, or PR-AUC.
- **PCA-style features (`V1`–`V28`)** are roughly zero-centered and bell-shaped, but several have heavy tails that may benefit from robust scaling or clipping.
- **`Amount`** is highly skewed with a long right tail; a `log1p` transform is likely worth trying before linear models.
- **Inter-feature correlation is low** between the `V*` columns (they are PCA components by construction); the strongest correlations are between a handful of `V*` features and the target, making them the most discriminative signals.
- **No missing values** were detected, so imputation is not required and the pipeline can focus on scaling, resampling, and model selection.